In [ ]:
!pip install -q -U datasets transformers accelerate scikit-learn underthesea "pyarrow<24"

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, concatenate_datasets, load_dataset
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)
from underthesea import word_tokenize

# ===================== CẤU HÌNH =====================
NEWS_REPO = "trieunh/vietnamese.human.ai.news.sets"
LIT_REPO = "trieunh/Vietnamese.Ai.Human.Literary.works.VuTrongPhung"
PARQUET_PATH = "train.parquet"
N_PAIRS = 8000                   # số cặp (người, AI) -> 2 * N_PAIRS mẫu; tăng nếu muốn
MAX_CHARS = 1200                 # cắt văn bản (PhoBERT chỉ đọc 256 token)
MIN_CHARS = 200
MAX_LEN = 256
EPOCHS = 3
BATCH = 16
LR = 2e-5
SEED = 42
MODEL_NAME = "vinai/phobert-base"
OUT_DIR = "results"
# ====================================================

set_seed(SEED)
print("GPU:", torch.cuda.is_available())

GPU: True


In [ ]:
def find_ai_col(df):
    cols = [c for c in df.columns if c.endswith("generated_text")]
    if not cols:
        cols = [c for c in df.columns if "generat" in c.lower()]
    if not cols:
        raise ValueError(f"Không tìm thấy cột văn bản AI. Các cột hiện có: {list(df.columns)}")
    return cols[0]


def clean(s):
    return " ".join(str(s).split())[:MAX_CHARS]


def make_pairs(df, n_pairs=None):
    # Mỗi dòng gốc -> 2 mẫu: bài người (0) và đoạn AI (1), cùng group_id.
    # Cột `text` của người bắt đầu bằng `user_prompt` còn đoạn AI thì không,
    # nên cắt `user_prompt` đi để hai lớp cùng là đoạn viết tiếp (tránh đường tắt).
    ai_col = find_ai_col(df)
    df = df.dropna(subset=["text", ai_col])
    if n_pairs and len(df) > n_pairs:
        df = df.sample(n=n_pairs, random_state=SEED)
    rows, skipped = [], 0
    for i, r in enumerate(df.to_dict("records")):
        human, ai = str(r["text"]), str(r[ai_col])
        prompt = r.get("user_prompt")
        if isinstance(prompt, str) and prompt:
            if not human.startswith(prompt):
                skipped += 1
                continue
            human = human[len(prompt):]
        human, ai = clean(human), clean(ai)
        if len(human) < MIN_CHARS or len(ai) < MIN_CHARS:
            skipped += 1
            continue
        gid = str(r.get("id", i))
        rows.append({"text": human, "label": 0, "group_id": gid})
        rows.append({"text": ai, "label": 1, "group_id": gid})
    print(f"Cột AI: {ai_col} | bỏ qua {skipped} dòng | tạo {len(rows)} mẫu")
    return pd.DataFrame(rows)


if os.path.exists(PARQUET_PATH):
    print("Đọc", PARQUET_PATH)
    raw = pd.read_parquet(PARQUET_PATH)
else:
    from huggingface_hub import hf_hub_download, list_repo_files
    files = [f for f in list_repo_files(NEWS_REPO, repo_type="dataset") if f.endswith(".parquet")]
    print("Các file parquet trong repo:", files)
    pick = (next((f for f in files if "train" in f.lower()), None)
            or next((f for f in files if "optithreshold" in f.lower()), None))
    if pick is None:
        raise FileNotFoundError(f"Không đoán được file train trong {files}. Gán PARQUET_PATH thủ công.")
    print("Dùng file:", pick)
    raw = pd.read_parquet(hf_hub_download(NEWS_REPO, pick, repo_type="dataset"))

data = make_pairs(raw, N_PAIRS)
print(data["label"].value_counts().to_string())
print("Độ dài ký tự trung bình theo nhãn:")
print(data.groupby("label")["text"].apply(lambda s: s.str.len().mean()).round(0).to_string())

Các file parquet trong repo: ['test.parquet', 'train.parquet', 'validation.parquet']
Dùng file: train.parquet


train.parquet: reconstructing file:   0%|          |  0.00B /  132MB            

train.parquet: downloading bytes:           |  0.00B            

Cột AI: sail-Sailor2-8B-Chat-generated_text | bỏ qua 40 dòng | tạo 15920 mẫu
label
0    7960
1    7960
Độ dài ký tự trung bình theo nhãn:
label
0    1133.0
1    1200.0


In [ ]:
# Chia 70/15/15 theo group_id (bản người và bản AI của cùng một bài nằm cùng một tập)
gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
tr_idx, rest_idx = next(gss.split(data, groups=data["group_id"]))
train_df, rest = data.iloc[tr_idx], data.iloc[rest_idx]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
v_idx, t_idx = next(gss2.split(rest, groups=rest["group_id"]))
val_df, test_df = rest.iloc[v_idx], rest.iloc[t_idx]
print(f"Train/Val/Test: {len(train_df)}/{len(val_df)}/{len(test_df)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tqdm.pandas()


def segment(text):
    # PhoBERT cần văn bản đã tách từ: "sinh viên" -> "sinh_viên"
    return word_tokenize(str(text), format="text")


def to_dataset(frame):
    frame = frame[["text", "label"]].copy()
    frame["text"] = frame["text"].progress_apply(segment)
    ds = Dataset.from_pandas(frame, preserve_index=False)
    return ds.map(
        lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN),
        batched=True,
        remove_columns=["text"],
    )


train_ds = to_dataset(train_df)
val_ds = to_dataset(val_df)
test_ds = to_dataset(test_df)

Train/Val/Test: 11136/2392/2392


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

  0%|          | 0/11136 [00:00<?, ?it/s]

Map:   0%|          | 0/11136 [00:00<?, ? examples/s]

  0%|          | 0/2392 [00:00<?, ?it/s]

Map:   0%|          | 0/2392 [00:00<?, ? examples/s]

  0%|          | 0/2392 [00:00<?, ?it/s]

Map:   0%|          | 0/2392 [00:00<?, ? examples/s]

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    try:
        auroc = roc_auc_score(labels, probs)
    except ValueError:
        auroc = 0.0
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="binary"),
        "auroc": auroc,
    }


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "Con người viết", 1: "AI tạo ra"},
    label2id={"Con người viết": 0, "AI tạo ra": 1},
)
steps_per_epoch = -(-len(train_ds) // BATCH)
warmup_steps = int(0.1 * steps_per_epoch * EPOCHS)
args = TrainingArguments(
    output_dir=OUT_DIR,
    learning_rate=LR,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="auroc",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)
trainer.train()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  543MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors: reconstructing file:   0%|          |  0.00B /  543MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch,Training Loss,Validation Loss,Accuracy,F1,Auroc
1,0.072699,1.645810,0.699415,0.768885,0.965192
2,0.032944,0.881861,0.858696,0.876100,0.992772
3,0.000329,1.063349,0.848244,0.868240,0.988974


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2088, training_loss=0.08885075856656749, metrics={'train_runtime': 517.0788, 'train_samples_per_second': 64.609, 'train_steps_per_second': 4.038, 'total_flos': 4394431513299840.0, 'train_loss': 0.08885075856656749, 'epoch': 3.0})

In [ ]:
res = trainer.evaluate(test_ds, metric_key_prefix="test")
print({k: round(v, 4) for k, v in res.items() if isinstance(v, float)})

trainer.save_model(f"{OUT_DIR}/best_model")
tokenizer.save_pretrained(f"{OUT_DIR}/best_model")

Training Loss,Validation Loss,Epoch,Accuracy,F1,Auroc
0.000329,0.789642,3,0.869565,0.884530,0.996891


{'test_loss': 0.7896, 'test_accuracy': 0.8696, 'test_f1': 0.8845, 'test_auroc': 0.9969}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('results/best_model/tokenizer_config.json',
 'results/best_model/vocab.txt',
 'results/best_model/bpe.codes',
 'results/best_model/added_tokens.json')

In [ ]:
from scipy.special import softmax

def probs_labels(ds):
    out = trainer.predict(ds)
    return softmax(out.predictions, axis=-1)[:, 1], out.label_ids

pv, yv = probs_labels(val_ds)
pt, yt = probs_labels(test_ds)

thresholds = np.linspace(0.05, 0.95, 91)
best_t = max(thresholds, key=lambda t: f1_score(yv, pv >= t))
print("Ngưỡng tốt nhất trên validation:", round(best_t, 2))

for name, p, y in [("val", pv, yv), ("test", pt, yt)]:
    pred = (p >= best_t).astype(int)
    print(name, "acc", round(accuracy_score(y, pred), 4), "f1", round(f1_score(y, pred), 4))
print("test ở ngưỡng 0.5: acc", round(accuracy_score(yt, pt >= 0.5), 4), "f1", round(f1_score(yt, pt >= 0.5), 4))

Ngưỡng tốt nhất trên validation: 0.95
val acc 0.88 f1 0.8928
test acc 0.8905 f1 0.9012
test ở ngưỡng 0.5: acc 0.8696 f1 0.8845


In [ ]:
from huggingface_hub import hf_hub_download

lit_files = {
    "gemma": "Gemma-3-12B-VuTrongPhung.parquet",
    "sailor": "Sailor2-8B-VuTrongPhung.parquet",
}

for name, fname in lit_files.items():
    path = hf_hub_download(repo_id=LIT_REPO, filename=fname, repo_type="dataset")
    raw_lit = pd.read_parquet(path)
    print(f"\n[{name}] các cột:", list(raw_lit.columns), "| số dòng:", len(raw_lit))

    pairs = make_pairs(raw_lit)          # tự nhận cột *generated_text*, cắt user_prompt
    ds_lit = to_dataset(pairs)

    out = trainer.predict(ds_lit)
    s = out.predictions[:, 1] - out.predictions[:, 0]   # điểm "AI" - "người"
    y = out.label_ids

    print(f"[{name}] AUROC: {roc_auc_score(y, s):.4f}")
    print(f"[{name}] ngưỡng 0   : acc {accuracy_score(y, s >= 0):.4f} | f1 {f1_score(y, s >= 0):.4f}")
    if "best_t" in globals():
        print(f"[{name}] ngưỡng {float(best_t):.2f}: acc {accuracy_score(y, s >= best_t):.4f} | f1 {f1_score(y, s >= best_t):.4f}")

Gemma-3-12B-VuTrongPhung.parquet: reconstructing file:   0%|          |  0.00B / 1.15MB            

Gemma-3-12B-VuTrongPhung.parquet: downloading bytes:           |  0.00B            


[gemma] các cột: ['text', 'system_prompt', 'user_prompt', 'id', 'google-gemma-3-12b-it-hf_generated_text_wo_prompt', 'gen_meta'] | số dòng: 369
Cột AI: google-gemma-3-12b-it-hf_generated_text_wo_prompt | bỏ qua 0 dòng | tạo 738 mẫu


  0%|          | 0/738 [00:00<?, ?it/s]

Map:   0%|          | 0/738 [00:00<?, ? examples/s]

[gemma] AUROC: 1.0000
[gemma] ngưỡng 0   : acc 0.9973 | f1 0.9973
[gemma] ngưỡng 0.95: acc 0.9973 | f1 0.9973


... (2 outputs left out)


[sailor] các cột: ['text', 'system_prompt', 'user_prompt', 'id', 'sail-Sailor2-8B-Chat-hf_generated_text_wo_prompt', 'gen_meta'] | số dòng: 369
Cột AI: sail-Sailor2-8B-Chat-hf_generated_text_wo_prompt | bỏ qua 0 dòng | tạo 738 mẫu


  0%|          | 0/738 [00:00<?, ?it/s]

Map:   0%|          | 0/738 [00:00<?, ? examples/s]

[sailor] AUROC: 1.0000
[sailor] ngưỡng 0   : acc 0.9973 | f1 0.9973
[sailor] ngưỡng 0.95: acc 0.9973 | f1 0.9973


In [ ]:
def predict(text):
    inputs = tokenizer(segment(text), return_tensors="pt", truncation=True, max_length=MAX_LEN)
    inputs = {k: v.to(trainer.model.device) for k, v in inputs.items()}
    trainer.model.eval()
    with torch.no_grad():
        probs = torch.softmax(trainer.model(**inputs).logits, dim=-1)[0]
    pred = int(torch.argmax(probs))
    return trainer.model.config.id2label[pred], float(probs[pred])


sample = "Trong bối cảnh hiện nay, việc ứng dụng trí tuệ nhân tạo vào giáo dục đang là xu hướng tất yếu."
print(predict(sample))

('Con người viết', 0.9996359348297119)


In [ ]:


# %% ===== Ô 1: tạo bộ kiểm tra Wikipedia tiếng Việt (người) + văn bản AI do Qwen viết =====
import gc
import re

from transformers import AutoModelForCausalLM, AutoTokenizer

N_WIKI = 200                        # số bài; tăng nếu muốn kết quả ổn định hơn (chậm hơn)
GEN_MODEL = "Qwen/Qwen2.5-3B-Instruct"   # LLM KHÁC với Sailor2 dùng lúc huấn luyện, không cần đăng nhập
GEN_BATCH = 8

# --- 1) Đoạn mở đầu Wikipedia tiếng Việt (phần người viết) ---
wiki = load_dataset("wikimedia/wikipedia", "20231101.vi", split="train", streaming=True)
wiki = wiki.shuffle(seed=SEED, buffer_size=20000)

items = []
for ex in wiki:
    title = ex["title"]
    if title.startswith("Danh sách") or "định hướng" in title:
        continue
    paras = [p.strip() for p in ex["text"].split("\n") if p.strip()]
    if not paras:
        continue
    intro = paras[0]
    if not (300 <= len(intro) <= 1500) or len(ex["text"]) < 1500:
        continue
    items.append({"id": str(ex["id"]), "title": title, "human": intro})
    if len(items) >= N_WIKI:
        break
print("Số bài lấy được:", len(items))

# --- 2) Nhờ LLM viết đoạn mở đầu cùng tiêu đề (phần AI) ---
gtok = AutoTokenizer.from_pretrained(GEN_MODEL, padding_side="left")
try:
    llm = AutoModelForCausalLM.from_pretrained(GEN_MODEL, dtype=torch.float16, device_map="auto")
except TypeError:  # bản transformers cũ
    llm = AutoModelForCausalLM.from_pretrained(GEN_MODEL, torch_dtype=torch.float16, device_map="auto")
llm.eval()


def build_prompt(title, n_words):
    msgs = [
        {"role": "system", "content": "Bạn là người đóng góp cho Wikipedia tiếng Việt. Chỉ viết bằng tiếng Việt."},
        {"role": "user", "content": (
            f'Hãy viết đoạn mở đầu cho bài Wikipedia tiếng Việt có tiêu đề "{title}". '
            f"Đoạn văn dài khoảng {n_words} từ. Chỉ trả về đoạn văn, không thêm tiêu đề hay lời giải thích."
        )},
    ]
    return gtok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


ai_texts = []
for i in tqdm(range(0, len(items), GEN_BATCH)):
    batch = items[i:i + GEN_BATCH]
    prompts = [build_prompt(b["title"], min(max(len(b["human"].split()), 40), 150)) for b in batch]
    enc = gtok(prompts, return_tensors="pt", padding=True).to(llm.device)
    with torch.no_grad():
        gen = llm.generate(
            **enc,
            max_new_tokens=450,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=gtok.pad_token_id or gtok.eos_token_id,
        )
    for g in gen[:, enc["input_ids"].shape[1]:]:
        ai_texts.append(gtok.decode(g, skip_special_tokens=True))

del llm
gc.collect()
torch.cuda.empty_cache()

# --- 3) Ghép cặp, làm sạch ---
cjk = re.compile(r"[぀-ヿ㐀-鿿가-힯]")  # loại văn bản lẫn chữ Hán/Nhật/Hàn
rows, dropped = [], 0
for it, ai in zip(items, ai_texts):
    ai = re.sub(r"[*#`_]+", "", ai)          # bỏ ký hiệu markdown để không thành "dấu hiệu" thừa
    ai = " ".join(ai.split())
    human = " ".join(it["human"].split())
    if cjk.search(ai) or len(ai) < MIN_CHARS:
        dropped += 1
        continue
    rows.append({"text": human[:MAX_CHARS], "label": 0, "group_id": it["id"]})
    rows.append({"text": ai[:MAX_CHARS], "label": 1, "group_id": it["id"]})

wiki_pairs = pd.DataFrame(rows)
wiki_pairs.to_csv("wiki_test.csv", index=False)
print(f"Bỏ {dropped} bài | tạo {len(wiki_pairs)} mẫu | đã lưu wiki_test.csv")
print("Độ dài ký tự trung bình theo nhãn:")
print(wiki_pairs.groupby("label")["text"].apply(lambda s: s.str.len().mean()).round(0).to_string())
print("\nVí dụ phần AI:\n", wiki_pairs[wiki_pairs.label == 1]["text"].iloc[0][:400])


# %% ===== Ô 2: chấm điểm mô hình trên bộ Wikipedia =====
ds_w = to_dataset(wiki_pairs)
out = trainer.predict(ds_w)
s = out.predictions[:, 1] - out.predictions[:, 0]   # điểm "AI" - "người"
y = out.label_ids

print(f"[wiki] AUROC: {roc_auc_score(y, s):.4f}")
print(f"[wiki] ngưỡng 0   : acc {accuracy_score(y, s >= 0):.4f} | f1 {f1_score(y, s >= 0):.4f}")
if "best_t" in globals():
    print(f"[wiki] ngưỡng {float(best_t):.2f}: acc {accuracy_score(y, s >= best_t):.4f} | f1 {f1_score(y, s >= best_t):.4f}")

# Kiểm tra "đường tắt" độ dài: nếu AUROC chỉ dựa vào độ dài đã cao thì kết quả chưa đáng tin
len_auc = roc_auc_score(y, wiki_pairs["text"].str.len().values)
print(f"[wiki] AUROC nếu chỉ dùng độ dài văn bản: {len_auc:.4f}  (gần 0.5 là tốt)")

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

Số bài lấy được: 200


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (7 outputs left out)

Bỏ 32 bài | tạo 336 mẫu | đã lưu wiki_test.csv
Độ dài ký tự trung bình theo nhãn:
label
0    569.0
1    382.0

Ví dụ phần AI:
 Nam Phố Thiệu Minh còn được biết đến với tên gọi khác là Phố Thiệu, nằm ở trung tâm thành phố Huế, miền Trung Việt Nam. Đây là một trong những con phố cổ kính và mang đậm dấu ấn lịch sử của Huế, thu hút nhiều du khách và người dân địa phương ghé thăm mỗi ngày.


  0%|          | 0/336 [00:00<?, ?it/s]

Map:   0%|          | 0/336 [00:00<?, ? examples/s]

[wiki] AUROC: 0.8946
[wiki] ngưỡng 0   : acc 0.7024 | f1 0.6000
[wiki] ngưỡng 0.95: acc 0.6935 | f1 0.5796
[wiki] AUROC nếu chỉ dùng độ dài văn bản: 0.2149  (gần 0.5 là tốt)


In [ ]:
# Ô thêm vào CUỐI notebook, sau ô chấm điểm Wikipedia.
# Cần còn trong bộ nhớ: trainer, to_dataset, wiki_pairs, s, y (từ ô chấm điểm Wikipedia trước đó).
from sklearn.metrics import confusion_matrix


def cut(text, n):
    """Cắt về tối đa n ký tự, bỏ phần từ bị cắt dở."""
    if len(text) <= n:
        return text
    if text[n] == " ":          # vị trí cắt đã nằm đúng cuối một từ
        return text[:n]
    return text[:n].rsplit(" ", 1)[0]


# 1) Ma trận nhầm lẫn của lần chấm trước (ngưỡng 0): hàng = nhãn thật, cột = dự đoán
print("Ma trận nhầm lẫn (hàng: người/AI thật, cột: đoán người/AI):")
print(confusion_matrix(y, (s >= 0).astype(int)))

# 2) Bộ KHỚP ĐỘ DÀI: mỗi cặp cắt cả hai đoạn về cùng độ dài
rows = []
for gid, g in wiki_pairs.groupby("group_id"):
    h = g[g.label == 0]["text"].iloc[0]
    a = g[g.label == 1]["text"].iloc[0]
    n = min(len(h), len(a))
    rows.append({"text": cut(h, n), "label": 0, "group_id": gid})
    rows.append({"text": cut(a, n), "label": 1, "group_id": gid})
wiki_matched = pd.DataFrame(rows)
print("\nĐộ dài trung bình sau khi khớp:")
print(wiki_matched.groupby("label")["text"].apply(lambda t: t.str.len().mean()).round(0).to_string())

out_m = trainer.predict(to_dataset(wiki_matched))
s_m = out_m.predictions[:, 1] - out_m.predictions[:, 0]
y_m = out_m.label_ids

print(f"\n[khớp độ dài] AUROC: {roc_auc_score(y_m, s_m):.4f}")
print(f"[khớp độ dài] AUROC nếu chỉ dùng độ dài: {roc_auc_score(y_m, wiki_matched['text'].str.len().values):.4f}  (gần 0.5 là tốt)")
print(f"[khớp độ dài] ngưỡng 0: acc {accuracy_score(y_m, s_m >= 0):.4f} | f1 {f1_score(y_m, s_m >= 0):.4f}")

# 3) Độ chính xác theo CẶP, không cần ngưỡng: trong mỗi cặp, điểm AI có cao hơn điểm người không?
piv = wiki_matched.assign(score=s_m).pivot(index="group_id", columns="label", values="score")
print(f"[khớp độ dài] tỉ lệ cặp mà AI được chấm cao hơn người: {(piv[1] > piv[0]).mean():.4f}  ({len(piv)} cặp)")

Ma trận nhầm lẫn (hàng: người/AI thật, cột: đoán người/AI):
[[161   7]
 [ 93  75]]

Độ dài trung bình sau khi khớp:
label
0    379.0
1    381.0


  0%|          | 0/336 [00:00<?, ?it/s]

Map:   0%|          | 0/336 [00:00<?, ? examples/s]


[khớp độ dài] AUROC: 0.8908
[khớp độ dài] AUROC nếu chỉ dùng độ dài: 0.5073  (gần 0.5 là tốt)
[khớp độ dài] ngưỡng 0: acc 0.6964 | f1 0.5952
[khớp độ dài] tỉ lệ cặp mà AI được chấm cao hơn người: 0.9226  (168 cặp)


In [ ]:
SAVE_DIR = "/content/drive/MyDrive/vi_ai_best_model"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
!ls -la /content/drive/MyDrive/vi_ai_best_model

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

total 529388
drwxr-xr-x 2 root root      4096 Nov  9 00:42 .
drwxr-xr-x 3 root root      4096 Nov  9 00:42 ..
-rw-r--r-- 1 root root        22 Nov  9 00:42 added_tokens.json
-rw-r--r-- 1 root root   1135173 Nov  9 00:42 bpe.codes
-rw-r--r-- 1 root root       990 Nov  9 00:42 config.json
-rw------- 1 root root 540023384 Nov  9 00:42 model.safetensors
-rw-r--r-- 1 root root      1205 Nov  9 00:42 tokenizer_config.json
-rw-r--r-- 1 root root      5201 Nov  9 00:42 training_args.bin
-rw-r--r-- 1 root root    895321 Nov  9 00:42 vocab.txt
